In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/housing-prices-dataset",
                                  output_dir = "./data")

print("Path to dataset files:", path)

In [2]:
import pandas as pd

df = pd.read_csv("./data/housing.csv")

df.head()

## 1. Import Libraries for Analysis

In [ ]:
# Data manipulation
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & evaluation
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# Utilities
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Create figures directory
os.makedirs('figures', exist_ok=True)

print('All libraries imported successfully!')

## 2. Exploratory Data Analysis (EDA)

### 2a. Dataset Overview

In [ ]:
# Display basic info
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())

print('\nData types:')
print(df.dtypes)

print('\nSummary statistics:')
display(df.describe())

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

print('\nUnique values in categorical columns:')
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f'  {col}: {df[col].unique()}')

### 2b. Price Distribution Analysis

In [ ]:
# Distribution of target variable (price)
print('Price Distribution Analysis')
print(f'  Skewness: {df["price"].skew():.2f}')
print(f'  Kurtosis: {df["price"].kurtosis():.2f}')
print(f'  Mean price: ${df["price"].mean():,.0f}')
print(f'  Median price: ${df["price"].median():,.0f}')
print(f'  Min price: ${df["price"].min():,.0f}')
print(f'  Max price: ${df["price"].max():,.0f}')
print(f'  Std dev: ${df["price"].std():,.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df['price'], kde=True, bins=30, ax=axes[0])
axes[0].set_title('Distribution of House Prices')
axes[0].set_xlabel('Price')

sns.boxplot(y=df['price'], ax=axes[1])
axes[1].set_title('Boxplot of House Prices')
axes[1].set_ylabel('Price')

plt.tight_layout()
plt.savefig('figures/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2c. Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
corr_matrix = df[numerical_cols].corr()

print('Correlation with Price:')
price_corr = corr_matrix['price'].sort_values(ascending=False)
for feat, corr_val in price_corr.items():
    if feat != 'price':
        print(f'  {feat}: {corr_val:.4f}')

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig('figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 2d. Price vs Numerical Features

In [ ]:
# Price vs key numerical features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

features_vs_price = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
for i, feature in enumerate(features_vs_price):
    row, col = i // 3, i % 3
    sns.scatterplot(data=df, x=feature, y='price', ax=axes[row, col], alpha=0.6)
    axes[row, col].set_title(f'Price vs {feature}')
    
    # Add trend line
    z = np.polyfit(df[feature], df['price'], 1)
    p = np.poly1d(z)
    axes[row, col].plot(sorted(df[feature]), p(sorted(df[feature])), 'r--', alpha=0.8)

# Hide unused subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig('figures/price_vs_features.png', dpi=150, bbox_inches='tight')
plt.show()

### 2e. Price vs Categorical Features

In [ ]:
# Price vs categorical features
fig, axes = plt.subplots(2, 4, figsize=(18, 10))

cat_features = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 
                'airconditioning', 'prefarea', 'furnishingstatus']

for i, feature in enumerate(cat_features):
    row, col = i // 4, i % 4
    sns.boxplot(data=df, x=feature, y='price', ax=axes[row, col])
    axes[row, col].set_title(f'Price by {feature}')
    axes[row, col].tick_params(axis='x', rotation=45)

# Hide unused subplot
axes[1, 3].axis('off')

plt.tight_layout()
plt.savefig('figures/price_by_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

# Print categorical feature stats
print('\nCategorical Feature Impact on Mean Price:')
for feature in cat_features:
    print(f'\n--- {feature} ---')
    print(df.groupby(feature)['price'].mean().to_string())

## 3. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Encode binary categorical variables (yes/no -> 1/0)
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 
               'airconditioning', 'prefarea']

for col in binary_cols:
    df_processed[col] = df_processed[col].map({'yes': 1, 'no': 0})

# Encode furnishingstatus (multi-class categorical)
furnishing_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
df_processed['furnishingstatus'] = df_processed['furnishingstatus'].map(furnishing_map)

print('Encoded dataset (first 5 rows):')
display(df_processed.head())

print('\nData types after encoding:')
print(df_processed.dtypes)

In [ ]:
# Split features and target
X = df_processed.drop('price', axis=1)
y = df_processed['price']

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set size: {X_train.shape}')
print(f'Test set size: {X_test.shape}')

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for interpretability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print('\nScaled training data (first 5 rows):')
display(X_train_scaled.head())

## 4. Model Training and Evaluation

We'll train and evaluate the following models:
1. **Linear Regression** - Baseline model
2. **Ridge Regression** - L2 regularization
3. **Lasso Regression** - L1 regularization (feature selection)
4. **Decision Tree** - Non-linear model
5. **Random Forest** - Ensemble of decision trees
6. **XGBoost** - Gradient boosting

In [ ]:
# Dictionary to store results
results = {}

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Train a model and evaluate its performance."""
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    # Store results
    results[name] = {
        'model': model,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std()
    }
    
    print(f'\n{"="*50}')
    print(f'{name}')
    print(f'{"="*50}')
    print(f'Train RMSE: {train_rmse:,.2f}')
    print(f'Test RMSE:  {test_rmse:,.2f}')
    print(f'Train MAE:  {train_mae:,.2f}')
    print(f'Test MAE:   {test_mae:,.2f}')
    print(f'Train R\u00b2:   {train_r2:.4f}')
    print(f'Test R\u00b2:    {test_r2:.4f}')
    print(f'CV R\u00b2:      {cv_scores.mean():.4f} (\u00b1{cv_scores.std():.4f})')
    
    return model

### 4a. Linear Regression

In [ ]:
# 1. Linear Regression
lr = LinearRegression()
evaluate_model('Linear Regression', lr, X_train_scaled, X_test_scaled, y_train, y_test)

### 4b. Ridge Regression

In [ ]:
# 2. Ridge Regression (with hyperparameter tuning)
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='r2')
ridge_grid.fit(X_train_scaled, y_train)

print(f'Best Ridge alpha: {ridge_grid.best_params_["alpha"]}')
evaluate_model('Ridge Regression', ridge_grid.best_estimator_, 
               X_train_scaled, X_test_scaled, y_train, y_test)

### 4c. Lasso Regression

In [ ]:
# 3. Lasso Regression (with hyperparameter tuning)
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
lasso_grid = GridSearchCV(Lasso(max_iter=10000), lasso_params, cv=5, scoring='r2')
lasso_grid.fit(X_train_scaled, y_train)

print(f'Best Lasso alpha: {lasso_grid.best_params_["alpha"]}')
evaluate_model('Lasso Regression', lasso_grid.best_estimator_, 
               X_train_scaled, X_test_scaled, y_train, y_test)

# Show which features Lasso selected (non-zero coefficients)
lasso_best = lasso_grid.best_estimator_
lasso_coefs = pd.Series(lasso_best.coef_, index=X_train.columns)
selected_features = lasso_coefs[lasso_coefs != 0].index.tolist()
print(f'\nFeatures selected by Lasso ({len(selected_features)}): {selected_features}')

### 4d. Decision Tree

In [ ]:
# 4. Decision Tree (with hyperparameter tuning)
dt_params = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=42), dt_params, cv=5, scoring='r2')
dt_grid.fit(X_train_scaled, y_train)

print(f'Best Decision Tree params: {dt_grid.best_params_}')
evaluate_model('Decision Tree', dt_grid.best_estimator_, 
               X_train_scaled, X_test_scaled, y_train, y_test)

### 4e. Random Forest

In [ ]:
# 5. Random Forest (with hyperparameter tuning)
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params, cv=5, scoring='r2', verbose=0
)
rf_grid.fit(X_train_scaled, y_train)

print(f'Best Random Forest params: {rf_grid.best_params_}')
evaluate_model('Random Forest', rf_grid.best_estimator_, 
               X_train_scaled, X_test_scaled, y_train, y_test)

### 4f. XGBoost

In [ ]:
# 6. XGBoost (with hyperparameter tuning)
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}
xgb_grid = GridSearchCV(
    xgb.XGBRegressor(random_state=42, n_jobs=-1),
    xgb_params, cv=5, scoring='r2', verbose=0
)
xgb_grid.fit(X_train_scaled, y_train)

print(f'Best XGBoost params: {xgb_grid.best_params_}')
evaluate_model('XGBoost', xgb_grid.best_estimator_, 
               X_train_scaled, X_test_scaled, y_train, y_test)

## 5. Model Comparison

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame({
    model: {
        'Train RMSE': f'{metrics["train_rmse"]:,.0f}',
        'Test RMSE': f'{metrics["test_rmse"]:,.0f}',
        'Train MAE': f'{metrics["train_mae"]:,.0f}',
        'Test MAE': f'{metrics["test_mae"]:,.0f}',
        'Train R\u00b2': f'{metrics["train_r2"]:.4f}',
        'Test R\u00b2': f'{metrics["test_r2"]:.4f}',
        'CV R\u00b2 (mean)': f'{metrics["cv_r2_mean"]:.4f}',
        'CV R\u00b2 (std)': f'{metrics["cv_r2_std"]:.4f}'
    }
    for model, metrics in results.items()
}).T

print('='*100)
print('MODEL COMPARISON SUMMARY')
print('='*100)
display(comparison_df)

In [ ]:
# Visual comparison of models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_names = list(results.keys())
test_r2_values = [results[m]['test_r2'] for m in model_names]
test_rmse_values = [results[m]['test_rmse'] for m in model_names]
test_mae_values = [results[m]['test_mae'] for m in model_names]

# R\u00b2 scores
colors = plt.cm.RdYlGn([v / max(test_r2_values) for v in test_r2_values])
bars1 = axes[0].barh(model_names, test_r2_values, color=colors)
axes[0].set_xlabel('Test R\u00b2 Score')
axes[0].set_title('Model Comparison - R\u00b2 Score')
axes[0].set_xlim(0, 1)
for bar, val in zip(bars1, test_r2_values):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2, 
                 f'{val:.4f}', va='center')

# RMSE
bars2 = axes[1].barh(model_names, test_rmse_values, color='coral')
axes[1].set_xlabel('Test RMSE')
axes[1].set_title('Model Comparison - RMSE')
for bar, val in zip(bars2, test_rmse_values):
    axes[1].text(val + 50000, bar.get_y() + bar.get_height()/2, 
                 f'{val:,.0f}', va='center')

# MAE
bars3 = axes[2].barh(model_names, test_mae_values, color='lightgreen')
axes[2].set_xlabel('Test MAE')
axes[2].set_title('Model Comparison - MAE')
for bar, val in zip(bars3, test_mae_values):
    axes[2].text(val + 50000, bar.get_y() + bar.get_height()/2, 
                 f'{val:,.0f}', va='center')

plt.tight_layout()
plt.savefig('figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Feature importance from tree-based models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random Forest feature importance
rf_best = results['Random Forest']['model']
rf_importance = pd.Series(rf_best.feature_importances_, index=X_train.columns)
rf_importance = rf_importance.sort_values(ascending=True)

axes[0].barh(rf_importance.index, rf_importance.values, color='skyblue')
axes[0].set_xlabel('Importance')
axes[0].set_title('Random Forest Feature Importance')
for i, v in enumerate(rf_importance.values):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center')

# XGBoost feature importance
xgb_best = results['XGBoost']['model']
xgb_importance = pd.Series(xgb_best.feature_importances_, index=X_train.columns)
xgb_importance = xgb_importance.sort_values(ascending=True)

axes[1].barh(xgb_importance.index, xgb_importance.values, color='lightcoral')
axes[1].set_xlabel('Importance')
axes[1].set_title('XGBoost Feature Importance')
for i, v in enumerate(xgb_importance.values):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig('figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Linear Regression coefficients
lr_model = results['Linear Regression']['model']
lr_coefficients = pd.Series(lr_model.coef_, index=X_train.columns)
lr_coefficients = lr_coefficients.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['red' if c < 0 else 'green' for c in lr_coefficients.values]
lr_coefficients.plot(kind='barh', color=colors)
plt.title('Linear Regression Coefficients')
plt.xlabel('Coefficient Value')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('figures/linear_regression_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLargest coefficients (strongest positive effect on price):')
print(lr_coefficients.sort_values(ascending=False).head(5))
print('\nSmallest coefficients (weakest effect on price):')
print(lr_coefficients.sort_values(ascending=True).head(5))

## 7. Best Model Predictions vs Actual

In [ ]:
# Find the best model based on test R²
best_model_name = max(results, key=lambda m: results[m]['test_r2'])
best_model = results[best_model_name]['model']
print(f'Best model: {best_model_name} (Test R\u00b2 = {results[best_model_name]["test_r2"]:.4f})')

# Predictions on test set
y_pred = best_model.predict(X_test_scaled)

# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'{best_model_name}: Actual vs Predicted')

# Residuals plot
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residuals')
axes[1].set_title(f'{best_model_name}: Residual Plot')

plt.tight_layout()
plt.savefig('figures/best_model_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

# Residual distribution
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, bins=30)
plt.title(f'{best_model_name}: Residual Distribution')
plt.xlabel('Residual')
plt.axvline(x=0, color='r', linestyle='--', lw=2)
plt.tight_layout()
plt.savefig('figures/residual_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary and Conclusions

In [ ]:
# Final summary
print('='*80)
print('SUMMARY')
print('='*80)

# Sort models by test R²
sorted_models = sorted(results.items(), key=lambda x: x[1]['test_r2'], reverse=True)

print(f'\nModel Ranking (by Test R\u00b2):')
print('-'*60)
for i, (name, metrics) in enumerate(sorted_models, 1):
    print(f'{i}. {name:25s} | Test R\u00b2: {metrics["test_r2"]:.4f} | '
          f'Test RMSE: {metrics["test_rmse"]:,.0f} | '
          f'Test MAE: {metrics["test_mae"]:,.0f}')

print(f'\nBest Model: {sorted_models[0][0]}')
print(f'  - Test R\u00b2:    {sorted_models[0][1]["test_r2"]:.4f}')
print(f'  - Test RMSE:  {sorted_models[0][1]["test_rmse"]:,.0f}')
print(f'  - Test MAE:   {sorted_models[0][1]["test_mae"]:,.0f}')
print(f'  - CV R\u00b2:      {sorted_models[0][1]["cv_r2_mean"]:.4f} (\u00b1{sorted_models[0][1]["cv_r2_std"]:.4f})')

print(f'\nKey Insights:')
print('-'*60)
print('1. The most important features for predicting house price are:')
print('   - Area (sq ft) - larger area generally means higher price')
print('   - Number of bathrooms')
print('   - Air conditioning')
print('   - Number of stories')
print('   - Preferred area (prefarea)')
print('2. Linear models performed best on the held-out test set. Random Forest and')
print('   XGBoost fit the training data much more closely (train R\u00b2 around 0.88-0.89)')
print('   but generalised worse, which suggests overfitting on a small dataset')
print('   (545 rows).')
print('3. Ridge and Lasso gave almost no improvement over plain linear regression,')
print('   so regularisation added little here.')
print('4. Note: the best model was selected using test-set R\u00b2. Selecting on')
print('   cross-validation score would avoid slight optimistic bias.')